# New Findings — 5 Deep Dive Analyses
Run AFTER 06_YouTube_Enrichment.ipynb is complete and df_clean is saved.

In [7]:
import pandas as pd
df = pd.read_csv('../02_Datasets/processed/youtube_influencers_enriched.csv')
print('Tier values:', df['tier'].unique())
print('Niche values:', df['niche'].unique())

Tier values: ['nano' 'macro' 'mega' 'micro']
Niche values: ['Beauty' 'Fashion' 'Fitness' 'Lifestyle' 'Film & Streaming' 'Food'
 'Travel' 'Gaming' 'Shopping']


## Analysis 1 — Niche × Tier Heatmap

In [8]:
print('=== NICHE × TIER — Median VPS ===')
pivot_vps = df_clean.pivot_table(
    values='view_per_subscriber',
    index='niche',
    columns='tier',
    aggfunc='median'
).round(3)
# Only use columns that actually exist
tier_order = [t for t in ['nano','micro','macro','mega'] if t in pivot_vps.columns]
pivot_vps = pivot_vps[tier_order]
print(pivot_vps)

print('\n=== NICHE × TIER — Median Engagement Rate ===')
pivot_eng = df_clean.pivot_table(
    values='engagement_rate',
    index='niche',
    columns='tier',
    aggfunc='median'
).round(4)
tier_order = [t for t in ['nano','micro','macro','mega'] if t in pivot_eng.columns]
pivot_eng = pivot_eng[tier_order]
print(pivot_eng)

print('\n=== BEST COMBINATION (highest VPS) ===')
stacked = pivot_vps.stack().reset_index()
stacked.columns = ['niche','tier','median_vps']
print(stacked.sort_values('median_vps', ascending=False).head(5))

=== NICHE × TIER — Median VPS ===
tier               nano  micro  macro   mega
niche                                       
beauty            2.043  0.843  0.499  0.387
fashion           2.487  0.935  0.419  0.478
film & streaming  1.318  0.669  0.393  0.256
fitness           1.154  0.318  0.306  0.194
food              1.790  0.500  0.409  0.381
gaming            1.349  0.475  0.386  0.270
lifestyle         1.292  0.828  0.659  0.407
shopping          1.295  1.027  0.625  0.390
travel            2.235  0.663  0.421  0.449

=== NICHE × TIER — Median Engagement Rate ===
tier               macro    mega
niche                           
beauty            0.0303  0.0343
fashion           0.0400  0.0412
film & streaming  0.0406  0.0230
fitness           0.0347  0.0337
food              0.0483  0.0276
gaming            0.0487  0.0288
lifestyle         0.0369  0.0354
shopping          0.0291  0.0213
travel            0.0316  0.0341

=== BEST COMBINATION (highest VPS) ===
      niche  tier  me

## Analysis 2 — Recency Premium

In [9]:
df_clean['activity_level'] = pd.cut(
    df_clean['days_since_last_upload'],
    bins=[0, 30, 90, 365],
    labels=['Very Active (≤30 days)', 'Active (31–90 days)', 'Moderate (91–365 days)']
)

print('=== RECENCY PREMIUM ===')
print('\nChannel count per activity level:')
print(df_clean['activity_level'].value_counts())

print('\nMedian VPS by activity level:')
print(df_clean.groupby('activity_level')['view_per_subscriber'].median().round(4))

print('\nMedian Engagement Rate by activity level:')
print(df_clean.groupby('activity_level')['engagement_rate'].median().round(4))

print('\nTop niche per activity level:')
for level in df_clean['activity_level'].cat.categories:
    subset = df_clean[df_clean['activity_level'] == level]
    top_niche = subset.groupby('niche')['view_per_subscriber'].median().idxmax()
    print(f'  {level}: {top_niche}')

=== RECENCY PREMIUM ===

Channel count per activity level:
activity_level
Very Active (≤30 days)    2158
Moderate (91–365 days)     640
Active (31–90 days)        444
Name: count, dtype: int64

Median VPS by activity level:
activity_level
Very Active (≤30 days)    0.6066
Active (31–90 days)       1.0789
Moderate (91–365 days)    1.4314
Name: view_per_subscriber, dtype: float64

Median Engagement Rate by activity level:
activity_level
Very Active (≤30 days)    0.0359
Active (31–90 days)       0.0266
Moderate (91–365 days)    0.0333
Name: engagement_rate, dtype: float64

Top niche per activity level:
  Very Active (≤30 days): lifestyle
  Active (31–90 days): fashion
  Moderate (91–365 days): fashion


/var/folders/y6/bx22r0xs67s5j705_zpg_02c0000gn/T/ipykernel_25783/2987899140.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('activity_level')['view_per_subscriber'].median().round(4))
/var/folders/y6/bx22r0xs67s5j705_zpg_02c0000gn/T/ipykernel_25783/2987899140.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('activity_level')['engagement_rate'].median().round(4))


## Analysis 3 — Ideal Creator Profile

In [13]:
vps_75 = df_clean['view_per_subscriber'].quantile(0.75)

ideal = df_clean[
    (df_clean['view_per_subscriber'] >= vps_75) &
    (df_clean['days_since_last_upload'] <= 30)
].copy()

print('=== IDEAL CREATOR PROFILE ===')
print(f'Definition: Top 25% VPS + uploaded within last 30 days')
print(f'VPS threshold: ≥{vps_75:.3f}')
print(f'Total ideal creators: {len(ideal)} ({len(ideal)/len(df_clean)*100:.1f}% of dataset)')

print('\nTop tiers:')
print(ideal['tier'].value_counts())

print('\nTop niches:')
print(ideal['niche'].value_counts().head(5))

print('\nTop countries:')
print(ideal['country'].value_counts().head(5))

print(f"\nMedian VPS: {ideal['view_per_subscriber'].median():.4f}")
print(f"Median subscribers: {ideal['subscribers'].median():,.0f}")
print(f"Median video count: {ideal['api_video_count'].median():.0f}")
print(f"Median days since upload: {ideal['days_since_last_upload'].median():.0f}")

=== IDEAL CREATOR PROFILE ===
Definition: Top 25% VPS + uploaded within last 30 days
VPS threshold: ≥1.655
Total ideal creators: 668 (14.8% of dataset)

Top tiers:
tier
nano     336
micro    148
macro    139
mega      45
Name: count, dtype: int64

Top niches:
niche
beauty              139
fashion             114
lifestyle            94
film & streaming     82
food                 61
Name: count, dtype: int64

Top countries:
country
Unknown    272
IN         136
US         115
DE          24
GB          21
Name: count, dtype: int64

Median VPS: 3.2050
Median subscribers: 9,785
Median video count: 152
Median days since upload: 2


## Analysis 4 — Content Volume vs Engagement

In [11]:
df_clean['content_volume'] = pd.cut(
    df_clean['api_video_count'],
    bins=[4, 20, 50, 100, 500, float('inf')],
    labels=['5–20 videos', '21–50 videos', '51–100 videos', '101–500 videos', '500+ videos']
)

print('=== CONTENT VOLUME vs ENGAGEMENT ===')
print('\nChannel count per bucket:')
print(df_clean['content_volume'].value_counts().sort_index())

print('\nMedian VPS by video count:')
print(df_clean.groupby('content_volume')['view_per_subscriber'].median().round(4))

print('\nMedian Engagement Rate by video count:')
print(df_clean.groupby('content_volume')['engagement_rate'].median().round(4))

print('\n→ Does more content = higher or lower engagement?')
g = df_clean.groupby('content_volume')['engagement_rate'].median()
print('Trend:', 'DECREASING ↓ (dilution effect)' if g.iloc[0] > g.iloc[-1] else 'INCREASING ↑')

=== CONTENT VOLUME vs ENGAGEMENT ===

Channel count per bucket:
content_volume
5–20 videos        138
21–50 videos       295
51–100 videos      370
101–500 videos    1867
500+ videos       1852
Name: count, dtype: int64

Median VPS by video count:
content_volume
5–20 videos       4.6709
21–50 videos      2.9130
51–100 videos     2.0184
101–500 videos    0.8965
500+ videos       0.2661
Name: view_per_subscriber, dtype: float64

Median Engagement Rate by video count:
content_volume
5–20 videos          NaN
21–50 videos      0.0320
51–100 videos     0.0299
101–500 videos    0.0322
500+ videos       0.0370
Name: engagement_rate, dtype: float64

→ Does more content = higher or lower engagement?
Trend: INCREASING ↑


/var/folders/y6/bx22r0xs67s5j705_zpg_02c0000gn/T/ipykernel_25783/2196474523.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('content_volume')['view_per_subscriber'].median().round(4))
/var/folders/y6/bx22r0xs67s5j705_zpg_02c0000gn/T/ipykernel_25783/2196474523.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('content_volume')['engagement_rate'].median().round(4))
/var/folders/y6/bx22r0xs67s5j705_zpg_02c0000gn/T/ipykernel_25783/2196474523.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pan

## Analysis 5 — The 4 Quadrants

In [14]:
vps_med = df_clean['view_per_subscriber'].median()
days_med = 30  # Fixed at 30 days to match Tableau dashboard threshold

def assign_quadrant(row):
    high_vps = row['view_per_subscriber'] >= vps_med
    recently_active = row['days_since_last_upload'] <= days_med
    if high_vps and recently_active:
        return 'Q1 — Dream Creators'
    elif high_vps and not recently_active:
        return 'Q2 — Evergreen Creators'
    elif not high_vps and recently_active:
        return 'Q3 — Active but Struggling'
    else:
        return 'Q4 — Fading Out'

df_clean['quadrant'] = df_clean.apply(assign_quadrant, axis=1)

print('=== 4 QUADRANT ANALYSIS ===')
print(f'VPS median used: {vps_med:.4f}')
print(f'Days since upload median used: {days_med:.0f} days')
print('\nChannel counts:')
print(df_clean['quadrant'].value_counts())

print('\n=== QUADRANT PROFILES ===')
for q in ['Q1 — Dream Creators','Q2 — Evergreen Creators','Q3 — Active but Struggling','Q4 — Fading Out']:
    s = df_clean[df_clean['quadrant'] == q]
    if len(s) == 0: continue
    print(f'\n{q} ({len(s)} channels)')
    print(f'  Top tier:    {s["tier"].value_counts().index[0]}')
    print(f'  Top niche:   {s["niche"].value_counts().index[0]}')
    print(f'  Top country: {s["country"].value_counts().index[0]}')
    print(f'  Median VPS:  {s["view_per_subscriber"].median():.4f}')
    print(f'  Median subs: {s["subscribers"].median():,.0f}')

=== 4 QUADRANT ANALYSIS ===
VPS median used: 0.6334
Days since upload median used: 3 days

Channel counts:
quadrant
Q3 — Active but Struggling    1375
Q2 — Evergreen Creators       1371
Q1 — Dream Creators            894
Q4 — Fading Out                887
Name: count, dtype: int64

=== QUADRANT PROFILES ===

Q1 — Dream Creators (894 channels)
  Top tier:    nano
  Top niche:   beauty
  Top country: Unknown
  Median VPS:  1.4281
  Median subs: 80,300

Q2 — Evergreen Creators (1371 channels)
  Top tier:    nano
  Top niche:   beauty
  Top country: Unknown
  Median VPS:  1.8851
  Median subs: 4,860

Q3 — Active but Struggling (1375 channels)
  Top tier:    mega
  Top niche:   fitness
  Top country: US
  Median VPS:  0.2297
  Median subs: 530,000

Q4 — Fading Out (887 channels)
  Top tier:    macro
  Top niche:   beauty
  Top country: US
  Median VPS:  0.3118
  Median subs: 162,000
